# Case Status Transform

This notebook applies the same long-format logic as `case_status_transform.py`, then reshapes that result into a wide case-level output.

It does the following:
1. Drops duplicate `(Batch Set Name, Batch Name)` pairs, keeping the earliest `Created On` row.
2. Adds `Delivered_to_VA` as the case-level terminal flag.
3. Adds `Date_Delivered_to_VA` from the earliest terminal `Created On` date.
4. Adds `Subcontractor` from the earliest `Created On` row for the case.
5. Optionally excludes `Batch Name` values found in one or more omit-list CSV files.
6. Builds a wide output with one row per `Batch Name`.

In [ ]:
import csv
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

In [ ]:
TERMINAL_BATCH_SET_NAMES = {
    "Case Closeout",
    "FastPass Closeout",
    "FastPass Credentialing",
    "HITL Unworkable",
}

SUBCONTRACTORS = ("XBP", "IRM", "TSS")
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S.%f %z"
MAX_BATCH_SETS = 7

INPUT_PATH = Path("input/")
LONG_OUTPUT_PATH = Path("output/HITL_billing_long.csv")
WIDE_OUTPUT_PATH = Path("output/HITL_billing_long.csv")
OMIT_CSV_PATHS = ['omit_files/HITL_Activity_Report_20260116.csv', 'omit_files/2026-04-02-XBP-Closed-Unworkable.csv',
                   'omit_files/FCS-HITL-2026-Feb-Unworkable-1.csv', 'omit_files/FCS-HITL-2026-Feb-Unworkable-2.csv',
                    'omit_files/HITL_Activity_Report_20251202.csv', 'omit_files/HITL_Activity_Report_20260102.csv']

In [ ]:
def resolve_input_csv_paths(input_path: Path) -> List[Path]:
    if input_path.is_file():
        return [input_path]
    if input_path.is_dir():
        csv_paths = sorted(path for path in input_path.rglob("*.csv") if path.is_file())
        if not csv_paths:
            raise ValueError(f"No CSV files were found under input folder: {input_path}")
        return csv_paths
    raise ValueError(f"Input path does not exist or is not readable: {input_path}")


def load_rows(input_path: Path) -> Tuple[List[Dict[str, str]], Sequence[str]]:
    rows: List[Dict[str, str]] = []
    fieldnames: List[str] = []
    fieldname_set = set()
    required = {"Batch Name", "Batch Set Name", "Created On"}

    for csv_path in resolve_input_csv_paths(input_path):
        with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            current_fieldnames = list(reader.fieldnames or [])
            missing = required.difference(current_fieldnames)
            if missing:
                raise ValueError(f"Input file is missing required columns {sorted(missing)}: {csv_path}")

            for fieldname in current_fieldnames:
                if fieldname not in fieldname_set:
                    fieldnames.append(fieldname)
                    fieldname_set.add(fieldname)

            for row in reader:
                normalized_row = {fieldname: row.get(fieldname, "") for fieldname in fieldnames}
                for fieldname in current_fieldnames:
                    normalized_row[fieldname] = row.get(fieldname, "")
                rows.append(normalized_row)

    return rows, fieldnames


def write_rows(csv_path: Path, rows: List[Dict[str, str]], fieldnames: Sequence[str]) -> None:
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_omitted_batch_names(csv_paths: Sequence[Path]) -> set[str]:
    omitted_batch_names: set[str] = set()
    for csv_path in csv_paths:
        with Path(csv_path).open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            fieldnames = set(reader.fieldnames or [])
            if "Batch Name" not in fieldnames:
                raise ValueError(f"Omit file is missing required column 'Batch Name': {csv_path}")
            for row in reader:
                batch_name = (row.get("Batch Name") or "").strip()
                if batch_name:
                    omitted_batch_names.add(batch_name)
    return omitted_batch_names


def parse_created_on(value: str) -> Optional[datetime]:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() == "NULL":
        return None
    return datetime.strptime(cleaned, TIMESTAMP_FORMAT)


def parse_completed_on(value: str) -> Optional[datetime]:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() == "NULL":
        return None
    return datetime.strptime(cleaned, TIMESTAMP_FORMAT)


def row_sort_key(row: Dict[str, str]) -> Tuple[bool, Optional[datetime], str]:
    created_ts = parse_created_on(row.get("Created On", ""))
    batch_set_name = row.get("Batch Set Name", "")
    return (created_ts is None, created_ts, batch_set_name)


def normalize_date_delivered_to_va(value: str) -> str:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() in {"NULL", "N/A"}:
        return ""
    if len(cleaned) == 10 and cleaned.count("-") == 2:
        return cleaned
    parsed = parse_created_on(cleaned)
    return parsed.date().isoformat() if parsed is not None else ""


def deduplicate_rows(rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    indexed_rows = []
    for source_order, row in enumerate(rows):
        indexed_rows.append(
            {
                **row,
                "_created_ts": parse_created_on(row.get("Created On", "")),
                "_source_order": source_order,
            }
        )

    indexed_rows.sort(
        key=lambda row: (
            row.get("Batch Set Name", ""),
            row.get("Batch Name", ""),
            row["_created_ts"] is None,
            row["_created_ts"],
            row["_source_order"],
        )
    )

    seen = set()
    deduped = []
    for row in indexed_rows:
        key = (row.get("Batch Set Name", ""), row.get("Batch Name", ""))
        if key in seen:
            continue
        seen.add(key)
        deduped.append({k: v for k, v in row.items() if not k.startswith("_")})
    return deduped


def compute_case_metadata(rows: List[Dict[str, str]]) -> Dict[str, Dict[str, str]]:
    rows_by_batch_name: Dict[str, List[Dict[str, str]]] = {}
    for row in rows:
        rows_by_batch_name.setdefault(row.get("Batch Name", ""), []).append(row)

    metadata_by_batch_name: Dict[str, Dict[str, str]] = {}
    for batch_name, case_rows in rows_by_batch_name.items():
        terminal_rows = [
            row for row in case_rows if row.get("Batch Set Name", "") in TERMINAL_BATCH_SET_NAMES
        ]
        terminus = "1" if terminal_rows else "0"
        delivered_to_va_date = ""
        subcontractor = "N/A"

        if terminal_rows:
            earliest_terminal_row = sorted(terminal_rows, key=row_sort_key)[0]
            earliest_terminal_created_on = earliest_terminal_row.get("Created On", "")
            delivered_to_va_date = normalize_date_delivered_to_va(earliest_terminal_created_on)

            earliest_case_row = sorted(case_rows, key=row_sort_key)[0]
            earliest_case_batch_set_name = earliest_case_row.get("Batch Set Name", "").upper()
            for subcontractor_name in SUBCONTRACTORS:
                if subcontractor_name in earliest_case_batch_set_name:
                    subcontractor = subcontractor_name
                    break

        metadata_by_batch_name[batch_name] = {
            "Delivered_to_VA": terminus,
            "Date_Delivered_to_VA": delivered_to_va_date,
            "Subcontractor": subcontractor,
        }

    return metadata_by_batch_name


def transform_rows(rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    deduped_rows = deduplicate_rows(rows)
    metadata_by_batch_name = compute_case_metadata(deduped_rows)

    transformed_rows = []
    for row in deduped_rows:
        enriched_row = dict(row)
        enriched_row.update(metadata_by_batch_name[row.get("Batch Name", "")])
        transformed_rows.append(enriched_row)
    return transformed_rows


def omit_rows_by_batch_name(rows: List[Dict[str, str]], omitted_batch_names: set[str]) -> List[Dict[str, str]]:
    if not omitted_batch_names:
        return rows
    return [row for row in rows if row.get("Batch Name", "") not in omitted_batch_names]


def wide_output_columns() -> List[str]:
    columns = ["Batch Name", "Delivered_to_VA", "Date_Delivered_to_VA", "Subcontractor"]
    for idx in range(1, MAX_BATCH_SETS + 1):
        columns.extend(
            [
                f"BS{idx} Name",
                f"BS{idx} Created On",
                f"BS{idx} Completed On",
            ]
        )
    return columns


def transform_long_to_wide(rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    ranked_rows = []
    for row in rows:
        ranked_rows.append(
            {
                **row,
                "_created_ts": parse_created_on(row.get("Created On", "")),
                "_completed_ts": parse_completed_on(row.get("Completed On", "")),
            }
        )

    ranked_rows.sort(
        key=lambda row: (
            row.get("Batch Name", ""),
            row["_completed_ts"] is None,
            row["_completed_ts"],
            row["_created_ts"] is None,
            row["_created_ts"],
            row.get("Batch Set Name", ""),
        )
    )

    grouped_rows: Dict[str, List[Dict[str, str]]] = {}
    for row in ranked_rows:
        grouped_rows.setdefault(row["Batch Name"], []).append(row)

    wide_rows = []
    for batch_name in sorted(grouped_rows):
        group = grouped_rows[batch_name][:MAX_BATCH_SETS]
        first_row = group[0]
        wide_row = {column: "" for column in wide_output_columns()}
        wide_row["Batch Name"] = batch_name
        wide_row["Delivered_to_VA"] = first_row.get("Delivered_to_VA", "")
        wide_row["Date_Delivered_to_VA"] = normalize_date_delivered_to_va(
            first_row.get("Date_Delivered_to_VA", "")
        )
        wide_row["Subcontractor"] = first_row.get("Subcontractor", "")

        for idx, row in enumerate(group, start=1):
            wide_row[f"BS{idx} Name"] = row.get("Batch Set Name", "")
            wide_row[f"BS{idx} Created On"] = row.get("Created On", "")
            wide_row[f"BS{idx} Completed On"] = row.get("Completed On", "")

        wide_rows.append(wide_row)

    return wide_rows


In [ ]:
rows, original_fieldnames = load_rows(INPUT_PATH)
deduped_rows = deduplicate_rows(rows)
transformed_rows = transform_rows(rows)
omitted_batch_names = load_omitted_batch_names([Path(path) for path in OMIT_CSV_PATHS])
transformed_rows = omit_rows_by_batch_name(transformed_rows, omitted_batch_names)
long_output_fieldnames = list(original_fieldnames) + [
    "Delivered_to_VA",
    "Date_Delivered_to_VA",
    "Subcontractor",
]
wide_rows = transform_long_to_wide(transformed_rows)
wide_fieldnames = wide_output_columns()

print(f"Input rows: {len(rows)}")
print(f"Rows after dedupe: {len(deduped_rows)}")
print(f"Rows after omit filter: {len(transformed_rows)}")
print(f"Wide rows: {len(wide_rows)}")
print(f"Long output path: {LONG_OUTPUT_PATH.resolve()}")
print(f"Wide output path: {WIDE_OUTPUT_PATH.resolve()}")

In [ ]:
transformed_rows[:5]

In [ ]:
wide_rows[:5]

In [ ]:
write_rows(LONG_OUTPUT_PATH, transformed_rows, long_output_fieldnames)
write_rows(WIDE_OUTPUT_PATH, wide_rows, wide_fieldnames)
print(f"Wrote long CSV to: {LONG_OUTPUT_PATH.resolve()}")
print(f"Wrote wide CSV to: {WIDE_OUTPUT_PATH.resolve()}")